这里说频率是指一个机场航班出现的频率

## 加载数据

数据已经预处理过了

In [4]:
import pandas as pd

# 加载数据
data = pd.read_csv('../../data-hh/my/hh_result/result_all.csv', dtype={'aircraft': str})

# 查看前几行数据，确保加载成功
print(data.head())

print(data.info())

         flt_no bd_type    cap aircraft  legs  leg_no  duration  pax  \
0  KgJrsp7Jd78=      窄体  132.0      319     1       1      1.07   25   
1  P9IRwar34h0=      窄体  189.0      321     1       1      1.38  151   
2  mJitm0UDfM4=      窄体  132.0      319     1       1      1.57   38   
3  jXr97M1wpn4=      窄体  132.0      319     1       1      1.58  109   
4  izjfHOxAho4=      窄体  132.0      319     1       1      1.80  124   

              a             b  ...  year  month  day  weekday  hour  minute  \
0  KNqX4/Q5Noc=  HexFWXqbb8I=  ...  2023     10    1        6    12      15   
1  AKQNtuL5r6Q=  Mv6HkAiSLUk=  ...  2023     10    1        6    20      10   
2  n465JzB8Rrw=  N4hmDZN/CJQ=  ...  2023     10    1        6    16      45   
3  N4hmDZN/CJQ=  n465JzB8Rrw=  ...  2023     10    1        6    19      40   
4  5t+HPO9Mu/w=  X5e5r3CS4OA=  ...  2023     10    1        6    13       5   

   second          from            to   unit_price  
0       0  KNqX4/Q5Noc=  HexFWXqbb8I=  

## 编码分类变量

### 统计不同城市的频率

In [5]:
import pandas as pd



# 使用字典存储城市的统计信息
city_dict = {}

# 遍历 data 数据框
for index, row in data.iterrows():
    for city in [row['from'], row['to']]:  # 遍历 'from' 和 'to' 两列城市
        if city in city_dict:
            # 如果城市已经存在，更新 count 和 ave_pax
            count, total_pax = city_dict[city]
            city_dict[city] = (count + 1, total_pax + row['pax'])
        else:
            # 如果城市未出现过，初始化该城市的数据
            city_dict[city] = (1, row['pax'])

# 创建 city_count 数据框
city_count = pd.DataFrame(
    [(city, count, total_pax / count) for city, (count, total_pax) in city_dict.items()],
    columns=['city', 'count', 'ave_pax']
)

# 输出 city_count 数据框
print(city_count)

             city   count     ave_pax
0    KNqX4/Q5Noc=   29448   93.032566
1    HexFWXqbb8I=    2515   77.192445
2    AKQNtuL5r6Q=  258666  122.900895
3    Mv6HkAiSLUk=   14270  110.718150
4    n465JzB8Rrw=    3548   76.338782
..            ...     ...         ...
247  pep3IBxbrEI=     159   54.339623
248  wWFYSnYVShg=    2768   50.237717
249  Qejm7YTJTI4=     178  108.219101
250  MgrS1LT2KO4=      91   83.593407
251  Eed3EdkMDqg=      49   78.122449

[252 rows x 3 columns]


In [6]:
# 创建一个城市到 ave_pax 的映射字典
city_map = city_count.set_index('city')['ave_pax'].to_dict()

# 用 city_map 中的 ave_pax 替换 data 中的城市列
columns_to_replace = ['from', 'to', 'a', 'b', 'c']

for col in columns_to_replace:
    data[col] = data[col].map(city_map)

# 输出替换后的 data 数据框
print(data)

               flt_no bd_type    cap aircraft  legs  leg_no  duration  pax  \
0        KgJrsp7Jd78=      窄体  132.0      319     1       1      1.07   25   
1        P9IRwar34h0=      窄体  189.0      321     1       1      1.38  151   
2        mJitm0UDfM4=      窄体  132.0      319     1       1      1.57   38   
3        jXr97M1wpn4=      窄体  132.0      319     1       1      1.58  109   
4        izjfHOxAho4=      窄体  132.0      319     1       1      1.80  124   
...               ...     ...    ...      ...   ...     ...       ...  ...   
5999220  BzUm4im0EqA=      窄体  152.0      320     1       1      3.03   99   
5999221  w+GXbx7u3EM=      窄体  152.0      320     1       1      2.75  127   
5999222  9ceSbo4suds=      窄体  152.0      320     1       1      3.48   66   
5999223  +Pv2ewi/JZY=      窄体  158.0      320     1       1      1.42  157   
5999224  WqHQlgk5y8c=      窄体  158.0      320     1       1      1.68   95   

                  a           b  ...  year  month  day  weekday

In [7]:
import joblib
from sklearn.preprocessing import LabelEncoder
import os

# 定义需要编码的分类特征
# categorical_columns = ['flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c', 'from', 'to']
categorical_columns = ['flt_no', 'bd_type', 'aircraft']

# 创建并应用 LabelEncoder
label_encoders = {}

# 创建保存编码器的文件夹（如果文件夹不存在）
save_folder = '../../data-hh/my/encoder/'
os.makedirs(save_folder, exist_ok=True)  # 如果文件夹已存在，不会报错

# 遍历每个分类特征，使用 LabelEncoder 对其进行编码
for col in categorical_columns:
    le = LabelEncoder()  # 创建一个 LabelEncoder 实例
    data[col] = le.fit_transform(data[col])  # 对训练数据中的分类特征进行编码
    label_encoders[col] = le  # 将每个特征的编码器保存到字典中，方便后续使用

    # 保存每个编码器到指定文件夹
    encoder_path = os.path.join(save_folder, f"{col}_encoder_all.pkl")  # 构建保存路径
    joblib.dump(le, encoder_path)  # 使用 joblib 将编码器保存为 pkl 文件
    print(f"{col} 的编码器已保存为 {encoder_path}")  # 输出保存的路径

flt_no 的编码器已保存为 ../../data-hh/my/encoder/flt_no_encoder_all.pkl
bd_type 的编码器已保存为 ../../data-hh/my/encoder/bd_type_encoder_all.pkl
aircraft 的编码器已保存为 ../../data-hh/my/encoder/aircraft_encoder_all.pkl


In [8]:
# 查看处理后的数据
print(data.head())
print(X_train.dtypes)


   flt_no  bd_type    cap  aircraft  legs  leg_no  duration  pax           a  \
0    3967        3  132.0         1     1       1      1.07   25   93.032566   
1    4751        3  189.0         7     1       1      1.38  151  122.900895   
2    8777        3  132.0         1     1       1      1.57   38   76.338782   
3    8313        3  132.0         1     1       1      1.58  109  120.925444   
4    8214        3  132.0         1     1       1      1.80  124  100.704512   

            b  ...  year  month  day  weekday  hour  minute  second  \
0   77.192445  ...  2023     10    1        6    12      15       0   
1  110.718150  ...  2023     10    1        6    20      10       0   
2  120.925444  ...  2023     10    1        6    16      45       0   
3   76.338782  ...  2023     10    1        6    19      40       0   
4  100.189374  ...  2023     10    1        6    13       5       0   

         from          to   unit_price  
0   93.032566   77.192445  1235.200000  
1  122.900

NameError: name 'X_train' is not defined

## 特征和目标分离
我们要预测的是pax字段，其他字段作为特征。

In [9]:
# 特征列
X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 'second', 'from', 'to','unit_price']]

# 目标列
y = data['pax']

## 训练XGBoost模型

In [11]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# 自定义 SMAPE 函数
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred)
    return np.mean(diff / denominator) * 100

# 自定义评估函数
def smape_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    smape_value = smape(y_true, y_pred)
    return 'SMAPE', smape_value

# 数据划分
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 输出数据集大小
print(f'训练集大小: {X_train.shape[0]}')
print(f'验证集大小: {X_val.shape[0]}')
print(f'测试集大小: {X_test.shape[0]}')

# 转换为 DMatrix 格式
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

# 设置参数
params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.01,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'alpha': 10
}

# 设置评估集
evals = [(dtrain, 'train'), (dval, 'validation')]

# 训练模型，使用自定义评估指标
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    early_stopping_rounds=10,
    custom_metric=smape_eval,  # 使用 custom_metric 参数
    verbose_eval=10 #隔多少轮显示一次
)

# 预测测试集
y_pred = model.predict(dtest)

# 测试集 SMAPE 评估
test_smape = smape(y_test, y_pred)
print(f'SMAPE on Test Set: {test_smape:.2f}%')

# 测试集 MSE 评估
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error on Test Set: {mse}')


训练集大小: 4799380
验证集大小: 599922
测试集大小: 599923
[0]	train-rmse:45.26575	train-SMAPE:35.69010	validation-rmse:45.28441	validation-SMAPE:35.70191
[10]	train-rmse:42.95216	train-SMAPE:34.27840	validation-rmse:42.97493	validation-SMAPE:34.29292
[20]	train-rmse:40.85515	train-SMAPE:32.96234	validation-rmse:40.88140	validation-SMAPE:32.97819
[30]	train-rmse:38.99214	train-SMAPE:31.76474	validation-rmse:39.02134	validation-SMAPE:31.78203
[40]	train-rmse:37.37054	train-SMAPE:30.68349	validation-rmse:37.40239	validation-SMAPE:30.70119
[50]	train-rmse:35.99032	train-SMAPE:29.72609	validation-rmse:36.02496	validation-SMAPE:29.74413
[60]	train-rmse:34.75211	train-SMAPE:28.82932	validation-rmse:34.78900	validation-SMAPE:28.84758
[70]	train-rmse:33.73670	train-SMAPE:28.05791	validation-rmse:33.77569	validation-SMAPE:28.07660
[80]	train-rmse:32.85214	train-SMAPE:27.35991	validation-rmse:32.89293	validation-SMAPE:27.37903
[90]	train-rmse:32.08834	train-SMAPE:26.74090	validation-rmse:32.13101	validation-SMA

[840]	train-rmse:26.18700	train-SMAPE:21.11202	validation-rmse:26.24680	validation-SMAPE:21.13411
[850]	train-rmse:26.17428	train-SMAPE:21.10187	validation-rmse:26.23437	validation-SMAPE:21.12401
[860]	train-rmse:26.16165	train-SMAPE:21.09220	validation-rmse:26.22169	validation-SMAPE:21.11422
[870]	train-rmse:26.14982	train-SMAPE:21.08308	validation-rmse:26.21008	validation-SMAPE:21.10525
[880]	train-rmse:26.13831	train-SMAPE:21.07383	validation-rmse:26.19856	validation-SMAPE:21.09592
[890]	train-rmse:26.12717	train-SMAPE:21.06529	validation-rmse:26.18754	validation-SMAPE:21.08731
[900]	train-rmse:26.11584	train-SMAPE:21.05612	validation-rmse:26.17628	validation-SMAPE:21.07811
[910]	train-rmse:26.10370	train-SMAPE:21.04692	validation-rmse:26.16426	validation-SMAPE:21.06901
[920]	train-rmse:26.09283	train-SMAPE:21.03865	validation-rmse:26.15345	validation-SMAPE:21.06078
[930]	train-rmse:26.08145	train-SMAPE:21.03041	validation-rmse:26.14211	validation-SMAPE:21.05246
[940]	train-rmse:26.

目前的情况是统计城市出现的频率，效果比使用标签编码好，这个也是有理由的，因为一个城市的航班次数越多，肯定说明需求多，航空公司才安排这么多人，所以说航班次数和客流直接也是存在关联的，所以增强了模型的效果

In [12]:
# 显示20条测试结果（真实值 vs 预测值）
test_results = list(zip(y_test.values[:100], y_pred[:100]))  # 真实值和预测值
print("\n20条测试结果（真实值 vs 预测值）:")
for i, (true_value, pred_value) in enumerate(test_results):
    print(f"第{i+1}条: 真实值={true_value}, 预测值={pred_value:.2f}")


20条测试结果（真实值 vs 预测值）:
第1条: 真实值=91, 预测值=105.67
第2条: 真实值=255, 预测值=219.72
第3条: 真实值=163, 预测值=128.69
第4条: 真实值=234, 预测值=204.37
第5条: 真实值=82, 预测值=117.53
第6条: 真实值=88, 预测值=85.36
第7条: 真实值=66, 预测值=76.95
第8条: 真实值=224, 预测值=196.02
第9条: 真实值=181, 预测值=162.21
第10条: 真实值=131, 预测值=116.16
第11条: 真实值=116, 预测值=145.74
第12条: 真实值=99, 预测值=124.15
第13条: 真实值=127, 预测值=120.25
第14条: 真实值=30, 预测值=69.01
第15条: 真实值=106, 预测值=94.26
第16条: 真实值=155, 预测值=131.61
第17条: 真实值=139, 预测值=152.13
第18条: 真实值=28, 预测值=60.77
第19条: 真实值=88, 预测值=115.92
第20条: 真实值=200, 预测值=181.18
第21条: 真实值=102, 预测值=137.07
第22条: 真实值=93, 预测值=104.85
第23条: 真实值=118, 预测值=131.78
第24条: 真实值=182, 预测值=136.43
第25条: 真实值=69, 预测值=116.15
第26条: 真实值=49, 预测值=61.88
第27条: 真实值=70, 预测值=72.18
第28条: 真实值=146, 预测值=121.76
第29条: 真实值=31, 预测值=42.34
第30条: 真实值=179, 预测值=160.23
第31条: 真实值=168, 预测值=157.45
第32条: 真实值=174, 预测值=190.11
第33条: 真实值=124, 预测值=108.07
第34条: 真实值=96, 预测值=75.76
第35条: 真实值=132, 预测值=113.23
第36条: 真实值=162, 预测值=118.61
第37条: 真实值=144, 预测值=129.72
第38条: 真实值=185, 预测值=183.97
第39条: 真实值=130, 预测值=104

似乎对于较小值预测存在误差

In [13]:
def calculate_smape(y_true, y_pred):
    """
    计算 Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))
    return smape

def calculate_mape(y_true, y_pred):
    """
    计算 Mean Absolute Percentage Error (MAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mape = 100 * np.mean(np.abs((y_true - y_pred) / y_true))
    return mape

In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 评估模型
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = calculate_mape(y_test, y_pred)
smape = calculate_smape(y_test, y_pred)

# 打印结果
print(f'Mean Squared Error (MSE): {mse:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Mean Absolute Percentage Error (MAPE): {mape:.4f}%')
print(f'Symmetric Mean Absolute Percentage Error (SMAPE): {smape:.4f}%')

Mean Squared Error (MSE): 675.3083
Root Mean Squared Error (RMSE): 25.9867
Mean Absolute Error (MAE): 20.4241
Mean Absolute Percentage Error (MAPE): 25.0472%
Symmetric Mean Absolute Percentage Error (SMAPE): 20.9628%


## 保存模型

In [ ]:
model.save_model("../../data-hh/my/模型文件/平均客流编码/xgboost_model_1000.json")
print("模型已保存为 xgboost_model.json")

## 超参数设置

In [1]:
import xgboost as xgb
import matplotlib.pyplot as plt

# 假设 model 是训练好的 XGBoost 模型
xgb.plot_importance(model, importance_type='weight', title="Feature Importance (Weight)", height=0.5)
plt.show()

xgb.plot_importance(model, importance_type='gain', title="Feature Importance (Gain)", height=0.5)
plt.show()

xgb.plot_importance(model, importance_type='cover', title="Feature Importance (Cover)", height=0.5)
plt.show()

NameError: name 'model' is not defined